# 08 — Gate 2a: scenarios_v2 + the S4 pilot (spec v0.11 §3.1, §7)

v0.11 resolves the places question WITHOUT touching the feature stack: **S4 = pure (w, t) on
the existing 8-feature stack** (m_soc t=0.552 + block-doubled carbon weights, exactly as
derived at Gate 1), and its places claim is *tested*, not assumed — one certified pilot solve
(~1 min) scored against a pre-registered band.

## Pre-registered acceptance band (FROZEN here, before the pilot runs)

**θ-tail mass capture ≥ 0.75 for BOTH carbon pools** (S0 reference: m_soc 0.435 / biomass
0.425; a1-at-w=1 reference: biomass 1.000). Rationale: the value statement is "a lot of the
tail," not completeness; S0's tail skip was a low-pull condition that carbon-forward reverses
by construction (block-doubled weights + at t=0.552 off-tail co-capture becomes
area-expensive).

- **PASS** → no formulation change anywhere; full stack symmetry; proceed to 09 (MGA).
- **FAIL** (either pool < 0.75) → STOP and take the numbers back to the chat. There is NO
  pre-authorized fallback: the tail-feature escalation was RESCINDED (Ethan, 2026-08-28) —
  tail masks will not enter the formulation as separate features.

Kernel: `R (y2y)`. Needs live internet (WLS).

In [1]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)
ctx   <- pr_setup(mpath, PROJ)

manifest refreshed from config.py (analysis=y2y)
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y


In [2]:
# ---- ingest (v0.11 critical path: the ORIGINAL 8-feature stack, no tails) ------------------
ctx <- modifyList(ctx, pr_ingest(ctx))
stopifnot("expected the original 8 continuous features (tails are contingency-only)" =
            ctx$n_cont == 8)
ctx <- modifyList(ctx, pr_planning_units(ctx))

ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 48 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget


In [3]:
# ---- scenarios_v2.json: v1 VERBATIM (S4 already = pure (w,t)) + v0.11 meta ----------------
v1 <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/scenarios_v1.json"))
sc <- v1
sc$`_meta`$spec_version <- "v0.11"
sc$`_meta`$estimator    <- "mga_maxham_v1"
sc$`_meta`$lineage      <- "scenarios_v1.json verbatim (S0-S4 (w,t) unchanged); v0.11 adds the S4 pilot + contingency"
sc$`_meta`$s4_acceptance_band <- list(
  statistic   = "theta-tail mass capture on the certified single S4 solve",
  min_capture = 0.75,
  pools       = c("irrecoverable_carbon_m_soc", "irrecoverable_carbon_biomass"),
  s0_reference = list(irrecoverable_carbon_m_soc = 0.435, irrecoverable_carbon_biomass = 0.425),
  on_fail     = "STOP -- back to the chat; tail-feature escalation RESCINDED (no pre-authorized fallback)")
sc$`_meta`$derived_utc_v2 <- format(Sys.time(), tz = "UTC")
# digits = 10: jsonlite defaults to 4 significant digits, which once truncated the
# shares/weights and broke the Gate-3 freeze (block shares summed to 1.0001).
jsonlite::write_json(sc, file.path(PROJ, "analyses/y2y/spec/scenarios_v2.json"),
                     auto_unbox = TRUE, pretty = TRUE, digits = 10)
cat("wrote analyses/y2y/spec/scenarios_v2.json\n")
S4W <- sc$S4_carbon$weights
S4T <- sc$S4_carbon$targets
cat("S4 targets:\n"); for (nm in names(S4T)) cat(sprintf("  %-32s %.3f\n", nm, as.numeric(S4T[[nm]])))
cat("S4 weight multipliers:\n")
for (nm in names(S4W)) cat(sprintf("  %-32s %.4f\n", nm, as.numeric(S4W[[nm]])))

wrote analyses/y2y/spec/scenarios_v2.json
S4 targets:
  irrecoverable_carbon_m_soc       0.552
S4 weight multipliers:
  climate_type_macrorefugia        1.2287
  transboundary_connectivity       0.5633
  climate_corridors                0.9858
  irrecoverable_carbon_m_soc       1.1659
  irrecoverable_carbon_biomass     0.5013
  aoh_richness_birds               1.1181
  aoh_richness_mammals             1.4370


In [4]:
# ---- the pilot: one certified single S4 solve (resumable) ----------------------------------
subdir <- "iter10_y2y_s4_pilot"
done <- file.path(PROJ, "output_data", subdir, "run_summary.json")
if (file.exists(done)) {
  cat(sprintf("== %s already solved -- skipped\n", subdir))
} else {
  actx <- pr_override(ctx, targets = S4T, feature_weight_multipliers = S4W,
                      results_subdir = subdir,
                      solver = "gurobi", decision_type = "binary",
                      opt_gap = 1e-4, portfolio_n = 1)
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing
  actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx))
  pr_write_outputs(actx)
}

  override targets          -> irrecoverable_carbon_m_soc=0.552
  override feature_weight_multipliers -> climate_type_macrorefugia=1.2287, transboundary_connectivity=0.563274, climate_corridors=0.985798, irrecoverable_carbon_m_soc=1.16587, irrecoverable_carbon_biomass=0.501302, aoh_richness_birds=1.11808, aoh_richness_mammals=1.43698
  override results_subdir   -> iter10_y2y_s4_pilot
  override solver           -> gurobi
  override decision_type    -> binary
  override opt_gap          -> 1e-04
  override portfolio_n      -> 1
  EFFECTIVE targets: irrecoverable_carbon_m_soc=0.552 | weight multipliers: climate_type_macrorefugia=1.2287, transboundary_connectivity=0.563274, climate_corridors=0.985798, irrecoverable_carbon_m_soc=1.16587, irrecoverable_carbon_biomass=0.501302, aoh_richness_birds=1.11808, aoh_richness_mammals=1.43698
  outputs  -> output_data/iter10_y2y_s4_pilot
weights: 8 continuous @ 1.0 ; 40 EFG @ 0.0250 (EFG group total = 1.0)
  up-weight climate_type_macrorefugia x1.2 -

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.552 and 1)
││└•weights:    continuous values (between 0.025 and 1.436979)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 49 rows, 1272962 columns and 16912559 nonzeros (Min)
Model fingerprint: 0xad85072f
Model has 48 linear objective coefficients
Variable types: 48 continuous, 1272914 integer (1272914 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [3e-02, 1e+00]
  Bounds range     [1e+00, 1e+0

In [5]:
# ---- score against the FROZEN band: theta-tail mass capture, both pools --------------------
consts <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/audit/audit_objects/audit_constants.json"))
theta <- as.numeric(consts$constants$theta)                 # frozen v0.3 (5x)
sel <- terra::rast(file.path(PROJ, "output_data/iter10_y2y_s4_pilot/portfolio.tif")) > 0.5
pools <- c("irrecoverable_carbon_m_soc", "irrecoverable_carbon_biomass")
score <- sapply(pools, function(p) {
  v <- terra::rast(file.path(PROJ, "input_data/aligned_stack", paste0(p, ".tif")))
  cut <- theta * terra::global(v, "mean", na.rm = TRUE)[[1]]      # stack layers are PU-masked
  tail <- v >= cut
  tot <- terra::global(v * tail, "sum", na.rm = TRUE)[[1]]
  cap <- terra::global(v * tail * sel, "sum", na.rm = TRUE)[[1]]
  cap / tot
})
rep <- read.csv(file.path(PROJ, "output_data/iter10_y2y_s4_pilot/portfolio_representation.csv"))
carb <- rep[rep$feature %in% pools, c("feature", "relative_held")]
cat("S4 pilot -- total capture:\n"); print(carb, row.names = FALSE)
cat(sprintf("\ntheta-tail mass capture (band >= 0.75 BOTH; S0 was 0.435 / 0.425):\n"))
for (p in pools) cat(sprintf("  %-32s %.3f  %s\n", p, score[[p]],
                             if (score[[p]] >= 0.75) "PASS" else "FAIL"))
PASS <- all(score >= 0.75)
verdict <- if (PASS) {
  "PASS -- no formulation change; full stack symmetry; proceed to 09_gate2b_mga"
} else {
  "FAIL -- STOP: no pre-authorized fallback (tail escalation RESCINDED); take the numbers to the chat"
}
cat(sprintf("\nS4 PILOT VERDICT: %s\n", verdict))
jsonlite::write_json(list(
  band = list(min_capture = 0.75, pools = pools),
  tail_capture = as.list(score), total_capture = setNames(as.list(carb$relative_held), carb$feature),
  s0_reference = list(irrecoverable_carbon_m_soc = 0.435, irrecoverable_carbon_biomass = 0.425),
  pass = PASS, verdict = verdict, created_utc = format(Sys.time(), tz = "UTC")),
  file.path(PROJ, "output_data/iter10_y2y_s4_pilot/pilot_score.json"),
  auto_unbox = TRUE, pretty = TRUE)
cat("wrote output_data/iter10_y2y_s4_pilot/pilot_score.json\n")

S4 pilot -- total capture:
                      feature relative_held
 irrecoverable_carbon_biomass     0.3286434
   irrecoverable_carbon_m_soc     0.5520033

theta-tail mass capture (band >= 0.75 BOTH; S0 was 0.435 / 0.425):
  irrecoverable_carbon_m_soc       0.960  PASS
  irrecoverable_carbon_biomass     0.772  PASS

S4 PILOT VERDICT: PASS -- no formulation change; full stack symmetry; proceed to 09_gate2b_mga
wrote output_data/iter10_y2y_s4_pilot/pilot_score.json


## Next

**PASS** → `09_gate2b_mga.ipynb` (the MGA reference run; ~2.5–3 h). **FAIL** → take the
numbers to the chat — no pre-authorized fallback exists (tail escalation rescinded).
Either way: pilot numbers → results_log R7 in the same session.